## 📚 Advanced RAG Pipeline for Personal CV/Resume

This notebook demonstrates a complete Retrieval-Augmented Generation (RAG) system for personal CV data:
- **LLM-based chunking** with structure (headline + summary + original text)
- **Vector embeddings** and semantic search
- **Reranking** to improve relevance
- **Query rewriting** for better retrieval
- **Final Q&A** with grounded answers

Each code block below has a detailed explanation to help you understand the flow.

In [91]:
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
from litellm import completion
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go
import os
import glob
import tiktoken
import numpy as np
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import ChatOllama

c:\Users\Ahmed Pasha\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:266: FutureWarning:

You are using a Python version (3.10.8) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.



### 📦 Step 1: Import Libraries

**What this does:**
- Imports all necessary dependencies for the RAG pipeline
- Includes: file handling, LLM clients, vector DB, embeddings, visualization, and LangChain utilities

**Why it matters:**
- Sets up the complete toolkit needed for document loading, chunking, embedding, retrieval, and answer generation
- Combines both native tools (OpenAI, ChromaDB) and LangChain abstractions for flexibility

In [2]:
load_dotenv(override=True)                                                                                  

False

### 🔐 Step 2: Load Environment Variables

**What this does:**
- Loads configuration from `.env` file (API keys, endpoints, etc.)

**Why it matters:**
- Keeps sensitive credentials out of code
- Makes the notebook GitHub-safe (don't commit `.env` files!)

In [3]:
DB_NAME = "data_base2"
collection_name = "docs"
embedding_model = "qwen3-embedding:8b"
KNOWLEDGE_BASE_PATH = Path("data2")
AVERAGE_CHUNK_SIZE = 450
MODEl="ollama/llama3.2:latest"
openai=OpenAI(base_url="http://localhost:11434/v1",api_key="ollama")

### ⚙️ Step 3: Configuration Settings

**What this does:**
- `DB_NAME`: ChromaDB storage folder name
- `collection_name`: Name of the vector collection
- `embedding_model`: Ollama embedding model (`qwen3-embedding:8b`)
- `KNOWLEDGE_BASE_PATH`: Folder containing CV markdown files (`data2/`)
- `AVERAGE_CHUNK_SIZE`: Target size for each chunk (~450 chars)
- `MODEl`: LLM model for chunking/reranking/answering
- `openai`: OpenAI-compatible client pointing to local Ollama

**Why it matters:**
- Centralizes all config in one place
- Easy to switch models, paths, or chunk sizes for experiments

In [4]:
# Inspired by LangChain's Document - let's have something similar

class Result(BaseModel):
    page_content: str
    metadata: dict

### 📄 Step 4: Define `Result` Schema

**What this does:**
- Creates a Pydantic model to represent a document/chunk
- `page_content`: The text content
- `metadata`: Source file path and document type

**Why it matters:**
- Provides a consistent format for retrieval results
- Similar to LangChain's `Document` class but simpler

In [5]:
# A class to perfectly represent a chunk

class Chunk(BaseModel):
    headline: str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
    summary: str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")
    original_text: str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way")

    def as_result(self, document):
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text,metadata=metadata)


class Chunks(BaseModel):
    chunks: list[Chunk]

### 🧩 Step 5: Define `Chunk` and `Chunks` Schemas

**What this does:**
- `Chunk`: Represents one piece of a document with:
  - `headline`: Short, query-friendly title
  - `summary`: 2-4 sentence overview
  - `original_text`: Exact text from the source
- `as_result()`: Converts a Chunk into a Result object
- `Chunks`: Container for multiple chunks (LLM response format)

**Why it matters:**
- Enforces structured output from the LLM
- Ensures every chunk has rich metadata + semantic summary
- Makes retrieval more effective than raw text splits

In [6]:
def fetch_documents():
    """A homemade version of the LangChain DirectoryLoader"""

    documents = []

    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append({"type": doc_type, "source": file.as_posix(), "text": f.read()})

    print(f"Loaded {len(documents)} documents")
    return documents

### 📂 Step 6: Document Loading Function

**What this does:**
- Walks through all subdirectories in `data2/`
- Reads every `.md` file recursively
- Stores each file's type (folder name), path, and full text

**Why it matters:**
- Converts filesystem structure into a clean list of documents
- Custom alternative to LangChain's `DirectoryLoader`
- Separates documents by type (cv, education, projects, etc.)

In [7]:
documents = fetch_documents()

Loaded 8 documents


### 📥 Step 7: Load All Documents

**What this does:**
- Executes the loading function
- Stores all documents in the `documents` list

**Why it matters:**
- This is the raw input corpus that will be chunked and indexed
- Shows how many documents were found

In [8]:
print(documents[5])

{'type': 'projects', 'source': 'data2/projects/projects.md', 'text': '# PROJECTS\n\nTotal Projects Completed: 4\n\n---\n\n## Project ID: PROJ-01\nTitle: RAG-Based AI Assistant  \nCategory: Retrieval-Augmented Generation (RAG) System  \nProject Link: https://github.com/ahmedpasha746666/insurellm-ai-assistant  \nStatus: Completed  \n\nOverview:\nDeveloped an end-to-end Retrieval-Augmented Generation (RAG) system that generates accurate, context-aware responses using a structured knowledge base instead of relying solely on pre-trained LLM knowledge.\n\nTechnology Stack:\n- Ollama\n- Llama 3.2 (3B, 128K context window)\n- ChromaDB\n- Transformers\n- nomic-embed-text (768-dimensional embedding model)\n\nCore Responsibilities:\n- Designed and implemented the full RAG pipeline.\n- Built query rewriting module for semantic optimization.\n- Generated vector embeddings for document chunks.\n- Performed similarity search using ChromaDB.\n- Implemented reranking logic for improved retrieval accura

### 🔍 Step 8: Inspect One Document

**What this does:**
- Prints document #5 to check structure

**Why it matters:**
- Quick validation of type/source/text formatting before expensive LLM processing

In [9]:
def make_prompt(document):
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    return f"""
You take a personal CV document and split it into overlapping chunks for a Resume Knowledge Base.

The document belongs to an individual and contains personal information such as:
- Education
- Projectsf
- Skills
- Experience
- Certifications
- Achievements

The document type is: {document["type"]}
The document source is: {document["source"]}

This knowledge base will be used by a chatbot to answer questions about the person’s profile, background, skills, and projects.

You must:
- Split the document into logical, meaningful chunks.
- Ensure the entire document is covered (do not omit any content).
- Create approximately {how_many} chunks (you may adjust if needed).
- Include about 20–30% overlap between chunks (around 50 words overlap) for better retrieval performance.

For each chunk, provide:
1. A clear headline
2. A short summary (2–4 sentences)
3. The exact original text of that chunk

Together, the chunks must represent the full document with overlap.

Here is the document:

{document["text"]}

Respond only with the structured chunks.
"""

### ✍️ Step 9: Build Chunking Prompt

**What this does:**
- Creates a detailed instruction prompt for the LLM
- Tells it to split CV documents into overlapping chunks
- Specifies the number of chunks based on document length
- Requires headline, summary, and original text for each chunk

**Why it matters:**
- Strong prompts → better chunk quality
- Overlap (20-30%) prevents losing context at boundaries
- Structured output makes parsing reliable

In [11]:
def make_message(document):
    return [
        {"role":"user","content":make_prompt(document)},
    ]

### 💬 Step 10: Format as Chat Message

**What this does:**
- Wraps the prompt into the messages format expected by LLM APIs
- Single user message with full instruction

**Why it matters:**
- Standardizes API call format for `litellm.completion()`

In [12]:
print(make_message(documents[0]))

[{'role': 'user', 'content': '\nYou take a personal CV document and split it into overlapping chunks for a Resume Knowledge Base.\n\nThe document belongs to an individual and contains personal information such as:\n- Education\n- Projects\n- Skills\n- Experience\n- Certifications\n- Achievements\n\nThe document type is: certification\nThe document source is: data2/certification/cert.md\n\nThis knowledge base will be used by a chatbot to answer questions about the person’s profile, background, skills, and projects.\n\nYou must:\n- Split the document into logical, meaningful chunks.\n- Ensure the entire document is covered (do not omit any content).\n- Create approximately 6 chunks (you may adjust if needed).\n- Include about 20–30% overlap between chunks (around 50 words overlap) for better retrieval performance.\n\nFor each chunk, provide:\n1. A clear headline\n2. A short summary (2–4 sentences)\n3. The exact original text of that chunk\n\nTogether, the chunks must represent the full d

### 👀 Step 11: Preview Message Payload

**What this does:**
- Shows what will be sent to the LLM for the first document

**Why it matters:**
- Verify prompts look correct before running expensive batch processing

In [13]:
def process_document(document):
    messages = make_message(document)
    response = completion(model=MODEl, messages=messages, response_format=Chunks)
    reply = response.choices[0].message.content
    doc_as_chunks = Chunks.model_validate_json(reply).chunks
    return [chunk.as_result(document) for chunk in doc_as_chunks]

### 🏭 Step 12: Process One Document into Chunks

**What this does:**
1. Build messages from document
2. Call LLM with structured output format (`Chunks`)
3. Parse JSON response into Pydantic objects
4. Convert each chunk to `Result` format with metadata

**Why it matters:**
- Single-document processor that can be scaled to full corpus
- Validates LLM output structure
- Preserves source/type metadata in each chunk

In [14]:
process_document(documents[0])

[Result(page_content='Certification ID: CERT-01\n\nData Science Certification from Kapil IT Skill Hub (April 2024 - September 2024). Focuses on machine learning, artificial neural networks, and practical implementation using Python, TensorFlow, and Keras.\n\n# CERT-01\nTitle: Data Science Certification \nCategory: Professional Certification \nInstitution: Kapil IT Skill Hub \nDuration: April 2024 – September 2024 \nStatus: Completed \nCertificate Link: https://drive.google.com/file/d/1jgsd2sTF9a4OkXqpBH2RDBcFfsoABrUH/view?usp=sharing \\n\nCore Competencies:\n- Machine Learning\n- Artificial Neural Networks (ANN)\n- Convolutional Neural Networks (CNN)\n- YOLO-based Object Detection\n- Natural Language Processing (NLP)\n- Large Language Models (LLMs)\\n\nPractical Implementation:\\n- Built ML models on real-world datasets.\n- Designed full ML lifecycle pipelines (data collection → preprocessing → feature engineering → training → validation → deployment).\n- Worked extensively with Python

c:\Users\Ahmed Pasha\AppData\Local\Programs\Python\Python310\lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 6: Expected `Message` - serialized value may not be as expected [field_name='message', input_value=Message(content='{ "chunk... reasoning_content=None), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='st...reasoning_content=None)), input_type=Choices])
  return self.__pydantic_serializer__.to_python(


### 🧪 Step 13: Test Single Document Processing

**What this does:**
- Runs chunking on one document to verify output

**Why it matters:**
- Catch errors/format issues before processing all documents
- See actual chunk output quality

In [15]:
def create_chunks(documents):
    chunks = []
    for doc in tqdm(documents):
        chunks.extend(process_document(doc))
    return chunks

### 🔁 Step 14: Batch Process All Documents

**What this does:**
- Loops through every document with a progress bar
- Calls `process_document()` on each
- Collects all chunks into one list

**Why it matters:**
- Scales single-document logic to the full corpus
- Progress bar shows completion status

In [16]:
chunks = create_chunks(documents)

  0%|          | 0/8 [00:00<?, ?it/s]c:\Users\Ahmed Pasha\AppData\Local\Programs\Python\Python310\lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 6: Expected `Message` - serialized value may not be as expected [field_name='message', input_value=Message(content='{\n\n  "... reasoning_content=None), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='st...reasoning_content=None)), input_type=Choices])
  return self.__pydantic_serializer__.to_python(
 38%|███▊      | 3/8 [03:53<05:45, 69.00s/it] c:\Users\Ahmed Pasha\AppData\Local\Programs\Python\Python310\lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 6: Expected `Message` - serialized value may not be as 

c:\Users\Ahmed Pasha\AppData\Local\Programs\Python\Python310\lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 6: Expected `Message` - serialized value may not be as expected [field_name='message', input_value=Message(content='{ "chunk... reasoning_content=None), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='st...reasoning_content=None)), input_type=Choices])
  return self.__pydantic_serializer__.to_python(


### ⚡ Step 15: Execute Chunk Generation

**What this does:**
- Runs full LLM-based chunking across all documents

**Why it matters:**
- **This is slow** (LLM calls for each doc)
- Produces the final chunk dataset for embedding

In [17]:
print(len(chunks))

69


### 📊 Step 16: Count Generated Chunks

**What this does:**
- Shows total number of chunks created

**Why it matters:**
- Validates chunking completed successfully
- Chunk count affects embedding costs

In [18]:
def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME)
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    texts = [chunk.page_content for chunk in chunks]
    emb = openai.embeddings.create(model=embedding_model, input=texts).data
    vectors = [e.embedding for e in emb]

    collection = chroma.get_or_create_collection(collection_name)

    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    print(f"Vectorstore created with {collection.count()} documents")

### 🗄️ Step 17: Create Vector Embeddings & Store

**What this does:**
1. Opens/creates ChromaDB at `DB_NAME`
2. Deletes old collection if it exists (fresh start)
3. Extracts text from chunks
4. Calls embedding model to get vectors
5. Stores ids, vectors, texts, and metadata in ChromaDB

**Why it matters:**
- Builds the semantic search index
- Enables similarity-based retrieval
- One-time setup (rerun only if documents change)

In [19]:
create_embeddings(chunks)

Vectorstore created with 69 documents


### 💾 Step 18: Execute Embedding & Storage

**What this does:**
- Runs the embedding creation process

**Why it matters:**
- **This takes time** (embedding all chunks)
- After this, you can query the vector database

In [118]:
DB_NAME="data_base"

### 🔄 Step 19: Switch Database Name

**What this does:**
- Updates `DB_NAME` to point to a different database

**Why it matters:**
- Useful if you want to load vectors from a different experiment
- **Important:** Make sure this DB exists before querying!

In [119]:
chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(collection_name)
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['type'] for metadata in metadatas]

# Robust color mapping (handles typo and unknown categories safely)
type_to_color = {
    'cv': 'blue',
    'education': 'green',
    'goals': 'red',
    'partime': 'orange',
    'technical concetps': 'yellow',  # folder typo in dataset
    'technical concepts': 'yellow',  # normalized spelling
}

colors = [type_to_color.get(t.strip().lower(), 'gray') for t in doc_types]

### 📊 Step 20: Load Vectors for Visualization

**What this does:**
1. Reconnects to ChromaDB
2. Retrieves embeddings, documents, and metadata
3. Converts to numpy arrays
4. Extracts document types
5. Maps types to colors (handles typos like "technical concetps")

**Why it matters:**
- Prepares data for 2D/3D visualization
- Robust color mapping prevents crashes from inconsistent naming

In [120]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

### 📈 Step 21: Visualize Embeddings in 2D

**What this does:**
1. Reduces high-dimensional vectors to 2D using t-SNE
2. Creates interactive scatter plot colored by document type
3. Hover shows type + text preview

**Why it matters:**
- Validates embedding quality visually
- Similar chunks should cluster together
- Helps debug retrieval issues

In [121]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

### 🌐 Step 22: Visualize Embeddings in 3D

**What this does:**
- Projects vectors into 3D space using t-SNE
- Creates interactive 3D scatter plot with rotation/zoom
- Same color scheme as 2D plot

**Why it matters:**
- Provides richer geometric view of separations/overlaps
- Easier to spot cluster patterns across categories
- Helpful for identifying embedding quality in higher dimensions

### Block 22 explanation (3D t-SNE plot)

**What this block does:** projects vectors into 3D and displays an interactive 3D scatter.
**Why it matters:** gives a richer geometry view to spot separations/overlaps across categories.

In [122]:
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )

### 🏆 Step 23: Define Reranking Schema

**What this does:**
- Creates `RankOrder` model to hold ordered chunk IDs
- Used for structured LLM output when reranking

**Why it matters:**
- Ensures LLM returns a valid list of integers
- Validates reranking output format

In [123]:
def rerank(question, chunks):
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    response = completion(model=MODEl, messages=messages, response_format=RankOrder)
    reply = response.choices[0].message.content
    order = RankOrder.model_validate_json(reply).order
    print(order)
    return [chunks[i - 1] for i in order]

### 🎯 Step 24: Rerank Retrieved Chunks

**What this does:**
1. Takes a question + initial retrieval results
2. Asks LLM to reorder chunks by true relevance
3. Returns chunks in improved order

**Why it matters:**
- Vector similarity alone can miss nuances
- LLM understands semantic relevance better
- Significantly improves final answer quality

In [124]:
RETRIEVAL_K = 10

def fetch_context_unranked(question):
    query = openai.embeddings.create(model=embedding_model, input=[question]).data[0].embedding
    results = collection.query(query_embeddings=[query], n_results=RETRIEVAL_K)
    chunks = []
    for result in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=result[0], metadata=result[1]))
    return chunks

### 🔎 Step 25: Basic Retrieval (Unranked)

**What this does:**
1. Embeds the user's question
2. Queries ChromaDB for top-K most similar chunks
3. Returns chunks without reranking

**Why it matters:**
- Raw vector search baseline
- Fast but not perfect relevance
- Used as input for reranking

In [125]:
question = "what is ahmed pasha contact number"
chunks = fetch_context_unranked(question)

### 🧪 Step 26: Test Basic Retrieval

**What this does:**
- Tests retrieval with a sample question
- Stores results in `chunks`

**Why it matters:**
- Validates retrieval is working
- Shows what raw search returns

In [126]:
question = "who is aqheel"
RETRIEVAL_K = 20
chunks = fetch_context_unranked(question)
for index, c in enumerate(chunks):
    if "manchester" in c.page_content.lower():
        print(index)

### 🔍 Step 27: Test Retrieval with Different Question

**What this does:**
- Retrieves top-20 chunks for "who is aqheel"
- Prints positions where "manchester" appears

**Why it matters:**
- Tests if relevant keywords show up in retrieval
- **Before reranking** — position shows raw similarity ranking

In [127]:
reranked = rerank(question, chunks)

[-1, -2, -3, -4, -5, -6, -7, -8, -9, 20, 19, 18, 17, 16, 15, 14, 13, 12, 11, 10]


c:\Users\Ahmed Pasha\AppData\Local\Programs\Python\Python310\lib\site-packages\pydantic\main.py:464: UserWarning:

Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 6: Expected `Message` - serialized value may not be as expected [field_name='message', input_value=Message(content='{ "order... reasoning_content=None), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='st...reasoning_content=None)), input_type=Choices])



### ⚡ Step 28: Apply Reranking

**What this does:**
- Reranks the retrieved chunks using LLM

**Why it matters:**
- Improves relevance order
- Printed `order` shows the new ranking

In [128]:
for index, c in enumerate(reranked):
    if "manchester" in c.page_content.lower():
        print(index)
        

### ✅ Step 29: Check Reranked Results

**What this does:**
- Prints new positions of "manchester" chunks **after reranking**

**Why it matters:**
- Compare with Step 27 to see reranking improvement
- Relevant keywords should move toward top

In [129]:
def fetch_context(question):
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)

### 🎁 Step 30: Retrieval + Reranking Wrapper

**What this does:**
- Combines retrieval and reranking in one function
- Always returns reranked results

**Why it matters:**
- Clean API for final QA system
- Hides complexity of two-step process

In [130]:
SYSTEM_PROMPT = """
You are a precise and professional AI assistant representing an individual’s personal CV and profile.

You are answering questions about the person's background, education, projects, skills, experience, certifications, and achievements.

CRITICAL INSTRUCTIONS:

1. Use ONLY the provided context extracts from the personal Knowledge Base.
2. Do NOT use outside knowledge.
3. Do NOT assume missing details.
4. Do NOT fabricate experience, skills, or qualifications.
5. If the answer is not clearly supported by the provided context, respond with:
   "I don't have enough information in the provided CV data to answer that accurately."

ANSWER REQUIREMENTS:

- Be accurate and factual.
- Be directly relevant to the question.
- Be complete but concise.
- If the question involves dates, numbers, or counts, extract them carefully from the context.
- If multiple sections are relevant (e.g., projects + skills), combine them clearly.

PERSONAL KNOWLEDGE BASE CONTEXT:
------------------------------------------------------------
{context}
------------------------------------------------------------

Now answer the user’s question using only the information above.
"""

### 📝 Step 31: Define System Prompt for QA

**What this does:**
- Creates detailed instruction template for answer generation
- Emphasizes accuracy, grounding in context, and honesty

**Critical rules:**
- Only use provided context
- Don't fabricate information
- Say "I don't know" if context is insufficient

**Why it matters:**
- Reduces hallucinations
- Ensures CV answers are factual
- Professional tone for resume assistant

In [131]:
# In the context, include the source of the chunk

def make_rag_messages(question, history, chunks):
    context = "\n\n".join(f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}" for chunk in chunks)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    return [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]

### 💬 Step 32: Build RAG Messages

**What this does:**
1. Formats retrieved chunks into context string
2. Injects context into system prompt
3. Builds full message array: [system + history + user question]

**Why it matters:**
- Gives LLM the evidence needed to answer
- Includes source paths for transparency
- Supports multi-turn conversation via history

In [132]:
def rewrite_query(question, history=[]):
    """Rewrite the user's question into a concise, highly specific search query 
    optimized for retrieving relevant information from a personal CV Knowledge Base."""
    
    message = f"""
You are assisting in retrieving information from a personal CV and profile Knowledge Base.

Your job is to transform the user's question into a short, precise, and retrieval-optimized search query.

CONTEXT:
The Knowledge Base contains structured information about:
- Education
- Technical skills
- Projects
- Work experience
- Certifications
- Tools and technologies
- Achievements

Conversation history:
{history}

User's current question:
{question}

INSTRUCTIONS:

1. Rewrite the question to maximize retrieval accuracy.
2. Make it very specific and focused.
3. Remove conversational words.
4. Keep important keywords (technologies, dates, project names, roles).
5. Do NOT add new information.
6. Do NOT answer the question.
7. Do NOT include explanations or extra text.

Respond ONLY with the refined Knowledge Base search query.
"""
    
    response = completion(
        model=MODEl,
        messages=[{"role": "system", "content": message}]
    )
    
    return response.choices[0].message.content.strip()

### 🔄 Step 33: Query Rewriting

**What this does:**
- Takes conversational user question
- Rewrites it into short, keyword-focused search query
- Uses LLM to extract core intent

**Why it matters:**
- User questions are often vague or chatty
- Dense queries improve retrieval accuracy
- Removes noise words, keeps important terms

In [133]:
def answer_question(question: str, history: list[dict] = []) -> tuple[str, list]:
    """
    Answer a question using RAG and return the answer and the retrieved context
    """
    query = rewrite_query(question, history)
    print(query)
    chunks = fetch_context(query)
    messages = make_rag_messages(question, history, chunks)
    response = completion(model=MODEl, messages=messages)
    return response.choices[0].message.content, chunks

### 🎯 Step 34: End-to-End Question Answering

**What this does (full RAG pipeline):**
1. **Rewrite** user question → focused query
2. **Retrieve** + **rerank** relevant chunks
3. **Generate** grounded answer with context
4. **Return** answer + source chunks

**Why it matters:**
- Complete RAG system in one function
- Returns both answer and evidence
- Prints rewritten query for transparency

In [134]:
answer_question("i want to know ahmed pashas work experience", [])

c:\Users\Ahmed Pasha\AppData\Local\Programs\Python\Python310\lib\site-packages\pydantic\main.py:464: UserWarning:

Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 6: Expected `Message` - serialized value may not be as expected [field_name='message', input_value=Message(content='"Ahmed P... reasoning_content=None), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='st...reasoning_content=None)), input_type=Choices])



"Ahmed Pasha work experience"


c:\Users\Ahmed Pasha\AppData\Local\Programs\Python\Python310\lib\site-packages\pydantic\main.py:464: UserWarning:

Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 6: Expected `Message` - serialized value may not be as expected [field_name='message', input_value=Message(content='{\n"orde... reasoning_content=None), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='st...reasoning_content=None)), input_type=Choices])



[17, 13, 18, 20, 16, 1, 15, 4, 12, 10, 19, 2, 14, 3]


c:\Users\Ahmed Pasha\AppData\Local\Programs\Python\Python310\lib\site-packages\pydantic\main.py:464: UserWarning:

Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 6: Expected `Message` - serialized value may not be as expected [field_name='message', input_value=Message(content='Accordin... reasoning_content=None), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='st...reasoning_content=None)), input_type=Choices])



('According to the provided context, Ahmed Pasha has 1 year of professional experience as an AI/ML Engineer at Pavaman Technologies, specializing in early plant disease prediction and computer vision systems.\n\nHere are the details of his work experience:\n\n- Total Professional Experience: 1 Year\n- Current Role: AI/ML Engineer\n- Employment Status: Full-Time\n- Company: Pavaman Technologies\n- Duration: May 2025 – Present',
 [Result(page_content='Objective\n\nTo build a grounded, context-aware AI assistant capable of answering insurance-related questions with high factual accuracy and minimal hallucination.\n\nTo build a grounded, context-aware AI assistant capable of answering insurance-related queries using a custom knowledge base instead of relying solely on pre-trained LLM knowledge.', metadata={'type': 'technical concetps', 'source': 'data2/technical concetps/insurellm.md'}),
  Result(page_content='Professional Summary\n\nAhmed Pasha is an AI/ML Engineer with 1 year of professi

### ✨ Step 35: Test - Ask About Work Experience

**What this does:**
- Tests the full QA system with a real question
- Returns answer + retrieved chunks

**Why it matters:**
- Validates the entire RAG pipeline works end-to-end
- Shows quality of final answers

In [141]:
answer_question("how many certificates does ahmed pasha have", [])

"CERTIFICATES - Ahmed Pasha"


c:\Users\Ahmed Pasha\AppData\Local\Programs\Python\Python310\lib\site-packages\pydantic\main.py:464: UserWarning:

Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 6: Expected `Message` - serialized value may not be as expected [field_name='message', input_value=Message(content='"CERTIFI... reasoning_content=None), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='st...reasoning_content=None)), input_type=Choices])



[20, 9, 11, 14, 4, 17, 5, 10, 16, 18, 19, 7, 13]


c:\Users\Ahmed Pasha\AppData\Local\Programs\Python\Python310\lib\site-packages\pydantic\main.py:464: UserWarning:

Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 6: Expected `Message` - serialized value may not be as expected [field_name='message', input_value=Message(content='{\n  "or... reasoning_content=None), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='st...reasoning_content=None)), input_type=Choices])

c:\Users\Ahmed Pasha\AppData\Local\Programs\Python\Python310\lib\site-packages\pydantic\main.py:464: UserWarning:

Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 6: Expected `Message` - serialized value may not be as expected [field_name='message', input_value=Message(content='Accordin... reasoning_content=None), input_type=Message])
  Pydantic

('According to the provided context, Ahmed Pasha has a total of 4 certifications completed.',
 [Result(page_content='Chunking Strategy\n\nBalanced retrieval precision and contextual completeness by choosing a chunk size of approximately 600 tokens with an overlap of around 100 tokens.\n\n- Chunk size: ~600 tokens\n- Overlap: ~100 tokens\n- Avoided breaking definitions or policy explanations\n- Maintained context continuity for financial clauses\nRationale:\nBalance between retrieval precision and contextual completeness.', metadata={'type': 'technical concetps', 'source': 'data2/technical concetps/insurellm.md'}),
  Result(page_content='Education Background - General Information\n\n\n\n---', metadata={'source': 'data2/education/education.md', 'type': 'education'}),
  Result(page_content="Certification Overview\n\nThis section provides an overview of the individual's certifications, including total certifications completed, institution, and duration.\n\n# CERTIFICATIONS\nTotal Certifica

### ✨ Step 36: Test - Ask About Certificates

**What this does:**
- Tests with a different question about certificates/links

**Why it matters:**
- Verifies RAG works across different query types
- Tests extraction of specific details (URLs, links)